In [ ]:
import sys
from joblib import load

sys.path.insert(0, '../../../../../..')
sys.path.insert(0, '../../../../../../../')

# performance imports for torch: torch kernel uses one core only.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch
from torch import optim

from reimplemented_approaches.proactive_conformance_checking.data_prep_split_encode_new import PrefixDatasetTabularFFN
from reimplemented_approaches.proactive_conformance_checking.ffn_models import FFNSeparateIDP
from reimplemented_approaches.proactive_conformance_checking.training import Training

In [ ]:
# Load encoders:
# Load prepared and encoded datasets
train_set_dict, val_set_dict, test_set_dict = PrefixDatasetTabularFFN.load_datasets(save_path="../../../data_preparation/Sepsis/separate/")

train_labels = sorted(train_set_dict.keys())

print(f"Loaded {len(train_labels)} deviation labels: {train_labels}")

encoders = load("../../../data_preparation/Sepsis/separate/encoders.pkl")
encoder_values = list(encoders.values())[0]
print("Encoders: ", encoder_values)

In [ ]:
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print("Using device:", device)

In [ ]:
# input: size: each label net has same input size
input_size = list(train_set_dict.values())[0].tensors[0].size(1)
# fully connected hidden 1
fc_hidden_1= 256
# fully connected hidden 2
fc_hidden2 = 256
# dropout probability
p_dropout = 0.1

def build_model():
    return FFNSeparateIDP(input_size=input_size,
                          fc_hidden_1=fc_hidden_1,
                          fc_hidden_2=fc_hidden2,
                          dropout=p_dropout,
                          device=device).to(device)

In [ ]:
# Training configuration
batch_size = 128
shuffle = True
epochs = 300  # 300 with early stopping (20% val from all train)
learning_rate = 0.0001

trained_label_models = {}
training_histories = {}

# Build an train for each label in the deviation set an own LSTM
for label in train_labels:
    
    print(f"\nTraining deviation label: {label}")
    train_set = train_set_dict[label]
    val_set = val_set_dict[label]
    
    model = build_model()
    
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    optimizer_values = {"optimizer": optimizer,
                        "epochs": epochs,
                        "mini_batches": batch_size,
                        "shuffle": shuffle}
    
    training = Training(model=model,
                        train_set=train_set,
                        val_set=val_set,
                        optimizer_values=optimizer_values,
                        device=device,
                        loss_mode="separate",
                        saving_path=f"./FFN_separate_IDP_{label}.pkl")
    
    history = training.train(mode='ffn')
    
    training_histories[label] = history
    trained_label_models[label] = f"./FFN_separate_IDP_{label}.pkl"

print("\nFinished training all labels.")